<a href="https://colab.research.google.com/github/red-gunslinger/Int-Comp/blob/main/Participacion_marzo_19.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Sofia Gabriela Aguilar 36019

In [1]:
import copy
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split

Load everything

In [2]:
torch.manual_seed(42)
np.random.seed(42)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

cpu


In [3]:
X, y = fetch_california_housing(return_X_y=True, as_frame=True)
feature_names = list(X.columns)

In [4]:
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.2, random_state=42
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.5, random_state=42
)

print("Train:", X_train.shape, y_train.shape)
print("Val:", X_val.shape, y_val.shape)
print("Test:", X_test.shape, y_test.shape)

Train: (16512, 8) (16512,)
Val: (2064, 8) (2064,)
Test: (2064, 8) (2064,)


In [5]:
excluded_cols = ["Latitude", "Longitude"]
processed_cols = [col for col in feature_names if col not in excluded_cols]

Q1 = X_train[processed_cols].quantile(0.25)
Q3 = X_train[processed_cols].quantile(0.75)
IQR = Q3 - Q1

lower_bounds = Q1 - 1.5 * IQR
upper_bounds = Q3 + 1.5 * IQR

In [6]:
train_mask = pd.Series(True, index=X_train.index)

for col in processed_cols:
    train_mask &= (X_train[col] >= lower_bounds[col]) & (X_train[col] <= upper_bounds[col])

X_train_filtered = X_train.loc[train_mask].copy()
y_train_filtered = y_train.loc[train_mask].copy()

print("Original train size:", X_train.shape)
print("Filtered train size:", X_train_filtered.shape)

Original train size: (16512, 8)
Filtered train size: (13460, 8)


In [7]:
means = X_train_filtered[processed_cols].mean()
stds = X_train_filtered[processed_cols].std()
stds[stds == 0] = 1.0

In [8]:
X_train_tensor = torch.tensor(X_train_filtered.values, dtype=torch.float32)
X_val_tensor = torch.tensor(X_val.values, dtype=torch.float32)
X_test_tensor = torch.tensor(X_test.values, dtype=torch.float32)

y_train_tensor = torch.tensor(y_train_filtered.values, dtype=torch.float32).view(-1, 1)
y_val_tensor = torch.tensor(y_val.values, dtype=torch.float32).view(-1, 1)
y_test_tensor = torch.tensor(y_test.values, dtype=torch.float32).view(-1, 1)

In [9]:
train_loader = DataLoader(TensorDataset(X_train_tensor, y_train_tensor), batch_size=128, shuffle=True)
val_loader = DataLoader(TensorDataset(X_val_tensor, y_val_tensor), batch_size=128, shuffle=False)
test_loader = DataLoader(TensorDataset(X_test_tensor, y_test_tensor), batch_size=128, shuffle=False)

In [10]:
class Preprocessor(nn.Module):
    def __init__(self, means, stds, feature_names, excluded_cols):
        super().__init__()

        processed_cols = [c for c in feature_names if c not in excluded_cols]
        proc_idx = [feature_names.index(c) for c in processed_cols]
        excl_idx = [feature_names.index(c) for c in excluded_cols]

        self.register_buffer("proc_idx", torch.tensor(proc_idx, dtype=torch.long))
        self.register_buffer("excl_idx", torch.tensor(excl_idx, dtype=torch.long))
        self.register_buffer("mean", torch.tensor(means[processed_cols].values, dtype=torch.float32))
        self.register_buffer("std", torch.tensor(stds[processed_cols].values, dtype=torch.float32))

    def forward(self, x):
        x_proc = x[:, self.proc_idx]
        x_excl = x[:, self.excl_idx]

        x_proc = (x_proc - self.mean) / self.std

        out = torch.empty_like(x)
        out[:, self.proc_idx] = x_proc
        out[:, self.excl_idx] = x_excl
        return out

In [11]:
class HousingModel(nn.Module):
    def __init__(self, hidden_layers, dropout=0.0):
        super().__init__()

        self.preprocess = Preprocessor(means, stds, feature_names, excluded_cols)

        layers = []
        in_features = len(feature_names)

        for h in hidden_layers:
            layers.append(nn.Linear(in_features, h))
            layers.append(nn.ReLU())
            if dropout > 0:
                layers.append(nn.Dropout(dropout))
            in_features = h

        layers.append(nn.Linear(in_features, 1))
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        x = self.preprocess(x)
        return self.net(x)

In [12]:
class HousingPreprocessor(nn.Module):
    def __init__(self, lower_bounds, upper_bounds, means, stds, feature_names, excluded_cols):
        super().__init__()

        self.feature_names = feature_names
        self.excluded_cols = excluded_cols
        self.processed_cols = [col for col in feature_names if col not in excluded_cols]

        self.proc_idx = torch.tensor(
            [feature_names.index(col) for col in self.processed_cols],
            dtype=torch.long
        )
        self.excl_idx = torch.tensor(
            [feature_names.index(col) for col in excluded_cols],
            dtype=torch.long
        )

        self.register_buffer(
            "lower_bounds",
            torch.tensor(lower_bounds[self.processed_cols].values, dtype=torch.float32)
        )
        self.register_buffer(
            "upper_bounds",
            torch.tensor(upper_bounds[self.processed_cols].values, dtype=torch.float32)
        )
        self.register_buffer(
            "means",
            torch.tensor(means[self.processed_cols].values, dtype=torch.float32)
        )
        self.register_buffer(
            "stds",
            torch.tensor(stds[self.processed_cols].values, dtype=torch.float32)
        )

    def forward(self, x):
        x_proc = x[:, self.proc_idx]
        x_excl = x[:, self.excl_idx]

        x_proc = torch.clamp(x_proc, min=self.lower_bounds, max=self.upper_bounds)
        x_proc = (x_proc - self.means) / self.stds

        output = torch.empty_like(x)
        output[:, self.proc_idx] = x_proc
        output[:, self.excl_idx] = x_excl

        return output

In [13]:
class HousingRegressor(nn.Module):
    def __init__(self, lower_bounds, upper_bounds, means, stds, feature_names, excluded_cols, hidden_layers, dropout=0.0):
        super().__init__()

        self.preprocessor = HousingPreprocessor(
            lower_bounds, upper_bounds, means, stds, feature_names, excluded_cols
        )

        layers = []
        input_dim = len(feature_names)

        for hidden_dim in hidden_layers:
            layers.append(nn.Linear(input_dim, hidden_dim))
            layers.append(nn.ReLU())
            if dropout > 0:
                layers.append(nn.Dropout(dropout))
            input_dim = hidden_dim

        layers.append(nn.Linear(input_dim, 1))
        self.network = nn.Sequential(*layers)

    def forward(self, x):
        x = self.preprocessor(x)
        return self.network(x)

In [14]:
def train_model(model, train_loader, val_loader, epochs=50, lr=0.001):
    criterion = nn.MSELoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)

    best_val_loss = float("inf")
    best_state = copy.deepcopy(model.state_dict())

    for epoch in range(epochs):
        model.train()
        for xb, yb in train_loader:
            xb, yb = xb.to(device), yb.to(device)
            optimizer.zero_grad()
            pred = model(xb)
            loss = criterion(pred, yb)
            loss.backward()
            optimizer.step()

        model.eval()
        val_losses = []
        with torch.no_grad():
            for xb, yb in val_loader:
                xb, yb = xb.to(device), yb.to(device)
                pred = model(xb)
                loss = criterion(pred, yb)
                val_losses.append(loss.item())

        avg_val_loss = np.mean(val_losses)
        if avg_val_loss < best_val_loss:
            best_val_loss = avg_val_loss
            best_state = copy.deepcopy(model.state_dict())

    model.load_state_dict(best_state)
    return model, best_val_loss

In [15]:
def evaluate_model(model, loader):
    model.eval()
    y_true = []
    y_pred = []

    with torch.no_grad():
        for xb, yb in loader:
            xb = xb.to(device)
            pred = model(xb).cpu()
            y_true.append(yb)
            y_pred.append(pred)

    y_true = torch.cat(y_true).numpy().flatten()
    y_pred = torch.cat(y_pred).numpy().flatten()

    mse = np.mean((y_true - y_pred) ** 2)
    rmse = np.sqrt(mse)
    mae = np.mean(np.abs(y_true - y_pred))

    return {"MAE": mae, "RMSE": rmse, "MSE": mse}

In [16]:
configs = {
    "Model 1": {"hidden_layers": [32], "dropout": 0.0, "lr": 0.001, "epochs": 50},
    "Model 2": {"hidden_layers": [64, 32], "dropout": 0.0, "lr": 0.001, "epochs": 50},
    "Model 3": {"hidden_layers": [128, 64], "dropout": 0.1, "lr": 0.0005, "epochs": 50}
}

In [17]:
results = {}
histories = {}
trained_models = {}

for name, config in configs.items():
    print(f"\nTraining {name}...")

    model = HousingRegressor(
        lower_bounds=lower_bounds,
        upper_bounds=upper_bounds,
        means=means,
        stds=stds,
        feature_names=feature_names,
        excluded_cols=excluded_cols,
        hidden_layers=config["hidden_layers"],
        dropout=config["dropout"]
    ).to(device)

    model, best_val_loss = train_model(
        model,
        train_loader,
        val_loader,
        epochs=config["epochs"],
        lr=config["lr"]
    )

    train_metrics = evaluate_model(model, train_loader)
    val_metrics = evaluate_model(model, val_loader)

    results[name] = {
     "Train MAE": train_metrics["MAE"],
      "Train RMSE": train_metrics["RMSE"],
     "Train MSE": train_metrics["MSE"],
     "Val MAE": val_metrics["MAE"],
      "Val RMSE": val_metrics["RMSE"],
      "Val MSE": val_metrics["MSE"],
     "Best Val Loss": best_val_loss
  }

    trained_models[name] = model


Training Model 1...

Training Model 2...

Training Model 3...


In [18]:
results_df = pd.DataFrame(results).T
results_df

,Train MAE,Train RMSE,Train MSE,Val MAE,Val RMSE,Val MSE,Best Val Loss
Model 1,0.509413,0.672792,0.452650,0.528086,0.718481,0.516215,0.544236
Model 2,0.472593,0.634418,0.402486,0.481593,0.664887,0.442074,0.458741
Model 3,0.469637,0.635242,0.403532,0.479715,0.670672,0.449801,0.467448


In [19]:
best_model_name = results_df["Val RMSE"].astype(float).idxmin()
best_model = trained_models[best_model_name]

print("Best model:", best_model_name)

Best model: Model 2


In [20]:
test_metrics = evaluate_model(best_model, test_loader)
pd.DataFrame([test_metrics], index=[best_model_name])

,MAE,RMSE,MSE
Model 2,0.487489,0.657325,0.432076
